# RPS-1: Strategy Notebook

Owner: strategy. Follows SPEC.ipynb v1.0. Implements decide(self_hands, opponent_hands, rng=None) from Section 3 of the spec.

Approach: Stage 2 is a two-by-two zero-sum game. The rows are the two hands we can keep, and the columns are the two hands the opponent can keep. We choose the probabilities that maximize our worst-case expected payoff (the maximin strategy). In a zero-sum game this is the equilibrium strategy, so it does not depend on guessing the opponent's next move.

In [ ]:
import numpy as np

# Copied from SPEC.ipynb Section 1. Do not edit these here; change them in the spec.
SHAPES = ['Rock', 'Paper', 'Scissors']
PAYOFF = {'win': 1, 'tie': 0, 'loss': -1}
SEED = 42
BEATS = {'Rock': 'Scissors', 'Scissors': 'Paper', 'Paper': 'Rock'}

## 1. Payoff for one round

Payoff is from our point of view: +1 if our kept hand beats theirs, 0 for a tie, -1 for a loss.

In [ ]:
def payoff(mine, theirs):
    if mine not in SHAPES or theirs not in SHAPES:
        raise ValueError('unknown shape: ' + str(mine) + ' or ' + str(theirs))
    if mine == theirs:
        return PAYOFF['tie']
    if BEATS[mine] == theirs:
        return PAYOFF['win']
    return PAYOFF['loss']

## 2. Payoff matrix for Stage 2

Rows are the two hands we can keep (in left-to-right order). Columns are the two hands the opponent can keep. Entry [i][j] is our payoff if we keep row i and they keep column j.

In [ ]:
def payoff_matrix(mine_shapes, theirs_shapes):
    return [[payoff(a, b) for b in theirs_shapes] for a in mine_shapes]

## 3. Solve the 2x2 game (maximin)

Let p be the probability that we keep row 0. Against opponent column j, our expected payoff is p * A[0][j] + (1 - p) * A[1][j]. The opponent picks the column that is worst for us, so our guaranteed payoff is the minimum over the two columns. That minimum is a piecewise linear function of p, so its maximum occurs at p = 0, at p = 1, or where the two column lines cross. We evaluate those candidates and keep the best.

In [ ]:
def solve_2x2(A):
    """Return (p, value): probability of keeping row 0, and the guaranteed expected payoff."""
    candidates = [0.0, 1.0]
    denom = A[0][0] - A[1][0] - A[0][1] + A[1][1]
    if denom != 0:
        p_cross = (A[1][1] - A[1][0]) / denom
        if 0.0 <= p_cross <= 1.0:
            candidates.append(p_cross)

    best_p, best_v = None, None
    for p in candidates:
        v = min(p * A[0][j] + (1 - p) * A[1][j] for j in (0, 1))
        if best_v is None or v > best_v + 1e-12:
            best_p, best_v = p, v
    return float(best_p), float(best_v)

## 4. decide()

Input: two hand records for us and two for the opponent (Section 2 of the spec), with our hands in left-to-right order. Output follows Section 3 of the spec. This notebook also returns expected_payoff, which is an extra key beyond the spec. The spec needs to be updated to include it.

Sampling: if the optimal strategy is mixed (0 < p < 1), the keep position is drawn at random from p. If it is pure, no draw is needed and sampled is False. When no rng is passed, a generator seeded with SEED is used so results are reproducible.

In [ ]:
def decide(self_hands, opponent_hands, rng=None):
    if len(self_hands) != 2 or len(opponent_hands) != 2:
        raise ValueError('each player must have exactly 2 hands')
    if self_hands[0]['position'] > self_hands[1]['position']:
        raise ValueError('self_hands must be in left-to-right order')

    A = payoff_matrix([h['shape'] for h in self_hands],
                      [h['shape'] for h in opponent_hands])
    p, value = solve_2x2(A)

    if 0.0 < p < 1.0:
        if rng is None:
            rng = np.random.default_rng(SEED)
        keep_position = 0 if rng.random() < p else 1
        sampled = True
    else:
        keep_position = 0 if p == 1.0 else 1
        sampled = False

    return {
        'keep_position': keep_position,
        'keep_shape': self_hands[keep_position]['shape'],
        'probabilities': {0: p, 1: 1.0 - p},
        'sampled': sampled,
        'expected_payoff': value,
    }

## 5. Check against the worked example in SPEC

Self shows Paper and Scissors. Opponent shows Paper and Rock. Expected: keep the Paper hand with probability 2/3, the Scissors hand with probability 1/3, and an expected payoff of 1/3.

In [ ]:
EXPECTED = {
    'probability_keep_position_0': 2 / 3,
    'probability_keep_position_1': 1 / 3,
    'expected_payoff': 1 / 3,
}

mine = [h for h in EXAMPLE_DETECTION if h['player'] == 'self']
theirs = [h for h in EXAMPLE_DETECTION if h['player'] == 'opponent']

result = decide(mine, theirs, rng=np.random.default_rng(SEED))
print(result)

assert abs(result['probabilities'][0] - EXPECTED['probability_keep_position_0']) < 1e-9
assert abs(result['probabilities'][1] - EXPECTED['probability_keep_position_1']) < 1e-9
assert abs(result['expected_payoff'] - EXPECTED['expected_payoff']) < 1e-9
print('Worked example matches the spec.')

## 6. Edge cases

- Identical shapes: if we show Rock and Rock, both rows are the same, so the choice of hand does not change the outcome.
- Mirror check: the solver is compared against a brute-force grid search on random hands. Any mismatch would point to a bug in the solver.

In [ ]:
def make_hands(shapes, player):
    return [{'player': player, 'shape': s, 'position': i, 'confidence': None}
            for i, s in enumerate(shapes)]

# Identical shapes: we keep Rock either way. Opponent keeps Paper, which beats Rock.
r = decide(make_hands(['Rock', 'Rock'], 'self'),
           make_hands(['Scissors', 'Paper'], 'opponent'))
print('Rock+Rock vs Scissors+Paper:', r)
assert r['expected_payoff'] == -1.0
assert r['sampled'] is False

def grid_value(A, n=4001):
    best = -10.0
    for p in np.linspace(0.0, 1.0, n):
        v = min(p * A[0][j] + (1 - p) * A[1][j] for j in (0, 1))
        best = max(best, v)
    return best

rng_test = np.random.default_rng(SEED)
worst_gap = 0.0
for _ in range(500):
    shapes = [str(s) for s in rng_test.choice(SHAPES, size=4)]
    A = payoff_matrix(shapes[:2], shapes[2:])
    _, v = solve_2x2(A)
    worst_gap = max(worst_gap, abs(v - grid_value(A)))

print('Worst gap between solver and grid search over 500 random cases:', worst_gap)
assert worst_gap < 1e-3

## 7. Sampling check

For a mixed strategy, repeated draws should follow the probabilities. Here we draw 10,000 times from the worked example and compare the frequency of keeping the Paper hand with 2/3.

In [ ]:
draw_rng = np.random.default_rng(SEED)
draws = [decide(mine, theirs, rng=draw_rng)['keep_position'] for _ in range(10000)]
freq_paper = 1 - np.mean(draws)   # keep_position 0 is the Paper hand
print('Observed frequency of keeping Paper:', round(freq_paper, 4), 'expected:', round(2 / 3, 4))
assert abs(freq_paper - 2 / 3) < 0.02

## 8. Open items

- The opponent-side strategy (spec exercise) is left for the team to derive. Verify it by hand, and check the strategy notebook against it.
- Add expected_payoff to the spec's decide() output in Section 3 (version bump to 1.1).
- decide() assumes exactly two hands per player, passed in left-to-right order. Detection must provide that order.